In [14]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
%matplotlib inline

from graphviz import Digraph
from IPython.display import Image, display
from IPython.display import Image, display, HTML

In [30]:
class Value:
  def __init__(self,data, _children=(), _op='', label='') -> None:
    self.data = data
    self.grad = 0.0
    self._prev = set(_children) # this keeps track of the parents of the returned data
    self._op = _op # keeps track of the operations we are using, and so we know what operation we did to the two parents to get the outputed data value
    self.label = label
    self._backward = lambda: None


  def __repr__(self) -> str:  # this allows us to print out a nice looking value and not like bit notation values
    return f"Value(data={self.data})"

  #say we wanna add a and b, we gotta make a function for add since python doesn't know how to add two Value objects
  def __add__(self,other) -> 'Value':
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+') # "+" here is addition like normal since we are using the actual data number not the Value object
  
    def _backward(): # has access to self and other due to being insde the add function
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __radd__(self, other) -> 'Value':
    return self * other

  def __mul__(self,other) -> 'Value': # must say "mul" and not "multiply"
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
  
    return out
  
  def __rmul__(self,other) -> 'Value':
    return self * other
  
  def __sub__(self,other) -> 'Value':
    return self + other*-1

  def __rsub__(self,other) -> 'Value':
    return self * other

  def __pow__(self,other) -> 'Value':
    assert isinstance(other, (float,int))
    out = Value(self.data**other, (self,), '**') # we asserted that other is a plain number for simplicity here. and thus the inputs are self only

    def _backward():
      self.grad += other*(self.data**(other-1)) * out.grad # again we know that other is a plain number. we also ignore the derivative with respect to the exponent bc we would beed a natual log. which is icky ig
    out._backward = _backward
    
    return out

  def __truediv__(self,other) -> 'Value':
    # out = Value(self.data/other.data, (self, other), '/') # we dont need to define anything new since we have * and **
    out = self * other**-1 # we dont need .data here since we are using operations we already define

    # do not need any backward code here bc we are using * and ** which we already defined

    return out

    
  def exp(self) -> 'Value': # this defines just e^x, this isnt general exponentition 
    out = Value(math.exp(self.data), (self,), 'exp')

    def _backward():
      self.grad += out.data * out.grad #the derivative of out.data (e^x) is e^x so the local derivative is the same as the output data
    out._backward = _backward

    return out

  def tanh(self) -> 'Value':
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')

    def _backward():
      self.grad += (1 - t**2)  * out.grad # local derivative * out.grad
    out._backward = _backward
  
    return out
  
  def backward(self):
    topo = [] # unsure why it is called topo and why we keep track of it honestly ****
    visited = set() # set visited we use to keep track of nodes we have already iterated over
    def build_topo(v): # function with arbitrary input v
      if v not in visited: # if v isnt visted yet, call the bellow on that node. if it is already visited, then move on to the next node
        visited.add(v) # count it as visited so we can keep track of which nodes aren't and are visited
        for child in v._prev: # checks if that node as children
          build_topo(child) # if so, calls the build topo function on that child(s)
        topo.append(v) # if there are no more children to call to, append that node to topo ****
    build_topo(self) #call build topo on that output of the neuron we want to interate backwards on

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()


In [5]:
!apt-get install -y graphviz

from graphviz import Digraph
from IPython.display import Image, display

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='png', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label=f"{{ {n.label} | data {n.data:.4f} | grad {n.grad:.4f} }}", shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot


E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [6]:
#inputs x1 and x2
x1 = Value(2.0, label='x1')
x2 = Value(1.0, label='x2')

#weights w1 and w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias of the neuron
b = Value(6.7, label='b')

#x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label='x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
# o = n.tanh()


e = (2*n).exp()
o = (e - 1)/(e + 1)

o.label = 'o'
o.backward()

diagram = draw_dot(o)
svg = diagram.pipe(format='svg').decode('utf-8')
display(HTML(f'<div style="overflow:auto; height:500px; background:white">{svg}</div>'))

In [7]:
x1 = torch.Tensor([2.0]).double() ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double() ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double() ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double() ; w2.requires_grad = True
b = torch.Tensor([6.88]).double() ; b.requires_grad = True
n = w1*x1 + w2*x2 + b
o = torch.tanh(n)

print(o.item())
o.backward()

print('x1', x1.grad.item())
print('x2', x2.grad.item())
print('w1', w1.grad.item())
print('w2', w2.grad.item())

0.7064193777288968
x1 -1.5029149883073547
x2 0.5009716627691182
w1 1.0019433255382364
w2 0.0


In [37]:
class Neuron:

    def __init__(self, nin): # takes the input and the number of inputs
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)] # the weights are a random number from -1 to 1 for each neuron input
        self.b = Value(random.uniform(-1,1)) # the bias added to the weights x input values

    def __call__(self, x): # call the neuron function with the inputed x values
        act = sum((w1*x1 for w1,x1 in zip(self.w,x)), self.b) # the function multiplies the weight and adds the bias
        out = act.tanh() # nonlinearity so the neurons can do complex math. remeber the squishing property of tanh.
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout): # takes the input, the number of inputs, and the number of outputs of the layer
        self.neurons = [Neuron(nin) for _ in range(nout)] # the layer is the neuron fuction run for each desired number of ouputs or neurons in that layer

    def __call__(self, x): # takes the input, the number of inputs, and the number of outputs of the layer
        outs = [n(x) for n in self.neurons] # run the n function for each input into the function
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:

    def __init__(self, nin, nouts): # takes in the input, the number of inputs and the number of layers
        sz = [nin] + nouts # joins the input layer to the outputs layers sizes/number of neurons in each layer.
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))] # then self.layers is the tuple pairs for Layer(3,4) *for three inputs and 4 neurons in that layer* for each output neuron layer, excluding the the inital layer

    def __call__(self, x):
        for layer in self.layers: # for each layer in self.layers
            x = layer(x) # run the layer fuctions which returns those outs for each layer
        return x # so we get each layers output 

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
        


In [40]:
x = [1.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1]) # n is the function for the full network
n(x) # return the MLP for those initial x values


Value(data=0.8342038890633253)

In [41]:
n.parameters()

[Value(data=-0.4513543254122141),
 Value(data=-0.8472357720231918),
 Value(data=-0.008931294955366598),
 Value(data=-0.367776304960169),
 Value(data=0.8645497430865361),
 Value(data=-0.45773826954036245),
 Value(data=0.7670655677684102),
 Value(data=0.9293245447364495),
 Value(data=0.5647321989154459),
 Value(data=0.4899736875387828),
 Value(data=0.2563801827771286),
 Value(data=0.31405758467366574),
 Value(data=-0.7953352586641445),
 Value(data=0.17752025754062428),
 Value(data=-0.9191219134575741),
 Value(data=-0.9192946600466847),
 Value(data=-0.9188960061284563),
 Value(data=-0.4251056377769482),
 Value(data=0.23942784338119738),
 Value(data=-0.6272849942681307),
 Value(data=0.8759443378164498),
 Value(data=0.5437135279043388),
 Value(data=0.4541672550376079),
 Value(data=-0.12004593084615878),
 Value(data=-0.7206569647372791),
 Value(data=0.45297401256809655),
 Value(data=0.9493008930907338),
 Value(data=-0.21335660189020045),
 Value(data=-0.4372878092184247),
 Value(data=-0.73487

In [ ]:
x = [1.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1]) # n is the function for the full network
n(x) # return the MLP for those initial x values


In [ ]:
xs = [ # the inputs into the 4 different sets of MLPs
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0] # desired targets

[Value(data=0.8720786892811229),
 Value(data=0.9017555135285527),
 Value(data=0.9001818225795966),
 Value(data=0.8706432540510677)]

In [66]:
for k in range(10):

    #forward pass
    ypred = [n(x) for x in xs] # the outputs for each of the 4 sets of inputs
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    #backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    #nudge/update
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)



0 2.249053933082275e-05
1 2.2488431540406903e-05
2 2.248632414323036e-05
3 2.2484217139185417e-05
4 2.2482110528160134e-05
5 2.248000431004581e-05
6 2.2477898484732627e-05
7 2.247579305211085e-05
8 2.2473688012069683e-05
9 2.247158336450047e-05
